# 제6장. AI가 지어낸 책을 믿어도 될까? — AI와 함께 읽기

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITPainter/korea-china-classics-text-mining/blob/main/notebooks/ch6_llm.ipynb)

4장에서 컴퓨터가 『장자』에서 찾아낸 ‘나비 꿈’ 구절을 이번에는 Claude에게 물어보고, 그 답을 4장의 방법으로 검증합니다. 이어서 다국어 임베딩으로 원문과 번역문을 정렬하고, AI 활용 기록을 남깁니다.

In [ ]:
# 실습 환경 준비 (Colab에서 처음 한 번 실행)
import sys
if "google.colab" in sys.modules:
    !pip -q install "kclit @ git+https://github.com/ITPainter/korea-china-classics-text-mining.git" "sentence-transformers" "anthropic>=2.8"
from kclit.viz import setup_korean_font
print("그래프 글꼴:", setup_korean_font())

## 준비: API 키 등록

Colab 왼쪽의 열쇠 아이콘(보안 비밀)에 `ANTHROPIC_API_KEY`를 등록하고 노트북 접근을 허용합니다. 키를 셀에 직접 적지 마세요. API 사용에는 비용이 듭니다.

In [ ]:
from kclit import llm
llm.load_api_key()

## 6.1 번역 초안과 출전 묻기

『구운몽』 16회에서 육관대사가 『장자』를 들려주는 대목입니다. 결과는 ‘초안’이고, 확신이 없는 곳은 [?]로 표시하도록 지시했습니다.

In [ ]:
passage = "莊周夢爲蝴蝶，蝴蝶又變爲莊周。\n莊周之夢爲蝴蝶耶？\n蝴蝶之夢爲莊周耶？\n終不能辨之！"
draft = llm.translate_draft(passage)
print(draft)

In [ ]:
notes = llm.allusion_candidates(passage)
print(notes)

**꼭 확인하세요:** AI가 내놓은 출전은 후보일 뿐입니다. 그럴듯한 책 이름도 존재하지 않을 수 있습니다. 다음 단계에서 컴퓨터와 원문으로 확인합니다.

## 6.2 컴퓨터와 원문으로 검증하기

4장과 같은 방법으로 이 구절이 『장자』 「齊物論」의 어느 문장과 겹치는지 찾습니다. AI가 말한 출전과 이 결과를 비교하세요.

In [ ]:
from kclit import classics, reuse, text
zhuangzi = text.split_sentences(classics.qiwulun())
check = reuse.find_reuse({"구운몽 16회 구절": passage.splitlines()}, {"莊子·齊物論": zhuangzi}, n=2, min_score=0.15, min_shared=4)
reuse.to_frame(check)[["target_line", "source_line", "score", "shared"]]

AI의 출전 후보가 위 결과와 같은지 비교합니다. 같다면 원문의 문장을 직접 읽어 글자를 대조하고(『장자』는 ‘胡蝶’, 『구운몽』은 ‘蝴蝶’), 다르거나 원문에서 찾을 수 없다면 환각을 의심하고 기록합니다. 컴퓨터가 찾은 것도 ‘읽어 볼 후보’입니다.

## 6.3 원문과 번역 맞추기

원문과 번역문을 문장 단위로 짝짓고, 짝이 없는 문장(생략·첨가)과 유사도가 낮은 짝(의역 의심)을 표시합니다. 아래 번역은 예시용으로 직접 쓴 것이며 일부러 마지막 문장을 뺐습니다.

In [ ]:
from kclit import embed, align
src = ["莊周夢爲蝴蝶，蝴蝶又變爲莊周。", "莊周之夢爲蝴蝶耶？", "蝴蝶之夢爲莊周耶？", "終不能辨之！"]
tgt = ["장주가 꿈에 나비가 되었는데, 나비는 다시 장주로 변했다.", "장주의 꿈이 나비가 된 것인가?", "나비의 꿈이 장주가 된 것인가?"]
m = embed.sentence_model()
sim = embed.cosine_matrix(embed.encode(m, src), embed.encode(m, tgt))
for s, t, score in align.align(sim):
    print(align.label((s, t, score)), [src[i] for i in s], [tgt[j] for j in t], score)

In [ ]:
# 합치기 기준(merge_penalty)을 낮추면 결과가 어떻게 달라질까?
for s, t, score in align.align(sim, merge_penalty=0.1):
    print(align.label((s, t, score)), [src[i] for i in s], [tgt[j] for j in t], score)

기준을 낮추면 마지막 문장을 앞 문장과 합쳐서 ‘대응’으로 처리합니다. 어떤 기준으로 돌렸는지 기록해 두세요.

## 6.4 AI를 믿어도 될까

AI를 쓴 작업은 모델, 프롬프트, 날짜, 검증 방법을 기록표에 남깁니다. 논문과 저서에 AI 활용 사실을 밝힐 때 근거가 됩니다.

In [ ]:
llm.log_use("ai_log.csv", "번역 초안", "구운몽 16회 장주 구절 번역", "원문 대조 후 직접 교정")
llm.log_use("ai_log.csv", "출전 후보", "구운몽 16회 장주 구절 출전", "컴퓨터 인용 탐지와 莊子 齊物論 원문 확인")
import pandas as pd; pd.read_csv("ai_log.csv")